# 13 · SkewerJAX and fake_spectra: where the time goes

How long the two codes take to compute the same spectra, stage by stage, on one target: **TNG50-2** (snapshot 25,
$z = 3.0$, 1.19 billion gas cells, 349 GB), a regular **grid of $128 \times 128$ skewers along $y$** (16,384 skewers),
fake_spectra's pixels of 1 km/s, and the lines **$\mathrm{Ly}\alpha$** and **Si III 1206**. The grids $x, z = iL/n$ nest;
on this machine fake_spectra runs them within 20 minutes up to $n = 256$ (996 s; $n = 512$ takes longer).
Here $n = 128$, half that side, because SkewerJAX's dense $\tau$ sum on the CPU would take about 6 h per run at
$256^2$; notebook 14 gives the slope of every stage against the number of skewers.
Four entries, each timed by stage on the same machine:

- **fake_spectra** (native, kernel `skewerjax-fs`, its OpenMP on every core);
- **SkewerJAX default** and **SkewerJAX matched** (every option at fake_spectra's value, notebook 12) on the **CPU**;
- **SkewerJAX default** on the **GPU**.

The stages: **setup** (the skewers), **I/O** (the snapshot reads), **cell selection** (the cells within reach of the
skewers), **gas physics** (temperatures, ion fractions), **pieces** (the deposition of the cells along the skewers) and the
**$\tau$ sum**. fake_spectra searches its particles, deposits them and sums $\tau$ inside its C engine: its selection and pieces
are in its $\tau$. What each code computes, and how close they are, is in notebook 12; how each stage scales with the
snapshot, the number of skewers, the pixel width and the sampling, in notebook 14.

In [ ]:
import os
N_CPU = os.cpu_count()                                   # logical CPUs of this machine
os.environ["OMP_NUM_THREADS"] = str(N_CPU)               # fake_spectra's OpenMP threads (its C engine)
import skewerjax.config as config                        # imports neither JAX nor the package: the devices come first
config.configure_devices(
    n_cpu_devices=1,          # the CPU as one JAX device: XLA's own threads use every core (sharding the skewers over
                              # N_CPU host devices measured no faster for these kernels; config.devices("cpu") either way)
    platforms="cuda,cpu",     # the GPU next to the CPU (config.devices("gpu") is empty without one)
)
import contextlib, io, logging, tempfile
logging.getLogger("jax._src.xla_bridge").setLevel(logging.CRITICAL)
import numpy as np, jax
import matplotlib.pyplot as plt
from skewerjax.utils.plotting import add_panel_letters, plot_stage_grid, use_backend, use_style
# use_backend()      # interactive figures (ipympl) in a live notebook; inline when executed headless or without ipympl
use_style()        # paper style: Tol's colour-blind-safe "bright" colours and colour maps, minimum font sizes

CPU, GPU = config.devices("cpu"), config.devices("gpu")
jax.config.update("jax_default_device", CPU[0])          # arrays on the host by default; only the GPU entry's tau goes to the GPU
cpu_model = next(l.split(":", 1)[1].strip() for l in open("/proc/cpuinfo") if l.startswith("model name"))
ram_GB = next(int(l.split()[1]) for l in open("/proc/meminfo") if l.startswith("MemTotal")) / 1e6
print(f"CPU: {cpu_model}, {N_CPU} logical CPUs, {ram_GB:.0f} GB RAM; JAX CPU devices {CPU}")
print(f"GPU: {[d.device_kind for d in GPU] or 'none'}; precision {config.get_precision()} (the package's default)")

## The set-up

The skewers are in **box mode** (`SkewerBundle.axis_aligned`): parallel skewers across the box share one path coordinate,
so one $a(s)$, solved once in float64 on the host with $a = a_{\rm snap}$ at the middle of the path, one frame and one
pixel grid. The pixels are fake_spectra's: $n_{\rm bins} = \lfloor v_{\max}/1\ \mathrm{km/s}\rfloor$,
$dv = v_{\max}/n_{\rm bins}$, $v_{\max} = (aH)_{\rm snap}L$. SkewerJAX samples each path every 0.25 km/s.

**Batches.** SkewerJAX takes the skewers in batches of 1024, each through the whole route (selection, gas physics,
pieces, $\tau$), so that the host memory (fake_spectra's chords of 4096 skewers need more than its 125 GB) and the GPU's
12 GB hold one batch at a time; the stage times add up over the
batches. Each batch reads the positions of every cell again (from the page cache after the first one).

**Protocol.** The **warm-up** runs the first batch once (JIT compilation, the numba kernels, the page cache) and is
reported apart; its compilation is the warm-up minus the same batch in the timed runs. Then `N_RUNS` **timed runs** over
every batch; the bars are their medians. Every JAX result is pulled to the host (`np.asarray`), which waits until it is
ready (`block_until_ready`). fake_spectra compiles nothing, and reads more than the RAM holds, so all its runs are
timed. Everything is in float32 (the package's precision; fake_spectra's C engine works in double).

In [ ]:
from skewerjax.core import LineBundle, SkewerBundle, SnapshotFrame, cell_pieces, compute_optical_depth
from skewerjax.simulations import ArepoHDF5Reader, gas_to_cells, select_cells_near_paths
from skewerjax.comparison.benchmark import RunClock, StageTimer
from skewerjax.comparison.scaling import cache_label, cache_state, nested_grid, reference_index
from skewerjax.utils.paths import DATA_DIR as DATA   # $SKEWERJAX_DATA (docs/installation.md)

In [ ]:
lines = LineBundle(["HI-LyA", "SiIII-1206"], ref_line_id="HI-LyA")
SPECIES = list(lines.species_keys)

In [ ]:
SIM, SNAP, AXIS, SPACING_KMS, N_GRID = "TNG50-2", 25, 2, 0.25, 128   # along y (axis 2); points every 0.25 km/s; N_GRID x N_GRID skewers
N_RUNS, N_RUNS_FS, BATCH = 3, 3, 1024                                  # timed runs (SkewerJAX after the warm-up; fake_spectra); skewers per batch
reader = ArepoHDF5Reader(f"{DATA}/{SIM}", SNAP)
H = reader.header
L, AH, a_snap = H.box_cMpc, H.velfac_kms_per_cMpc, H.a                 # [cMpc], (aH)_snap [km/s per cMpc]
nbins, dv = H.pixel_grid(1.0)                                          # fake_spectra's pixels
v = (np.arange(nbins) + 0.5) * dv                                      # [km/s], over [0, v_max)
V_PERIOD = nbins * dv
SNAPDIR = reader.files[0].parent                                       # for the page-cache state (fincore)
print(f"{SIM}, z = {H.redshift:.3f}, L = {L:.3f} cMpc, {H.n_gas:,} gas cells in {H.n_files} files; {nbins} pixels of {dv:.6f} km/s")

In [ ]:
clock = RunClock()                                    # entry -> stage -> seconds per run (run 0: the warm-up)
CPU_DEF, CPU_MAT, GPU_DEF, FS = "SkewerJAX default, CPU", "SkewerJAX matched, CPU", "SkewerJAX default, GPU", "fake_spectra"
X, Z = nested_grid(N_GRID, L)                          # x, z = i L / N_GRID [cMpc], x outer: the grids of notebook 14 nest
N_SK = X.size
REF = reference_index(N_GRID)                          # the skewer at (L/4, L/4), notebook 12's
BATCHES = [slice(a, min(a + BATCH, N_SK)) for a in range(0, N_SK, BATCH)]
print(f"{N_SK:,} skewers in {len(BATCHES)} batches of {BATCH}; the reference skewer {REF} at x = {X[REF]:.3f}, z = {Z[REF]:.3f} cMpc")

## fake_spectra

`Spectra` on the same sightlines (an axis and a point in its units, ckpc/$h$) and the same pixel width, then `get_tau`
for each line. Its stages are timed from outside, without editing it: the construction of `Spectra` (**setup**), every call
of its snapshot reader `snapshot_set.get_data` (**I/O**), every call of its C engine `_do_interpolation_work` (**$\tau$**: the
particles near the sightlines, their deposition and the sum), and the rest of `get_tau` (**gas physics**: temperatures,
neutral fractions, the CLOUDY table, on every particle of every file). It reads the snapshot once per line. Its save file
goes to a temporary folder, removed afterwards, and `reload_file=True` makes it recompute every time.

In [ ]:
from fake_spectra.spectra import Spectra
cofm = np.column_stack([X, np.zeros_like(X), Z]) / H.code_length_cMpc    # [ckpc/h]
axis = np.full(X.size, AXIS, dtype=np.int32)                              # 2 = y
fs_runs, fs_cache = [], []
for run in range(N_RUNS_FS):
    fs_cache.append(cache_state(SNAPDIR))                                 # the fraction of the snapshot in the page cache
    T = StageTimer()
    with tempfile.TemporaryDirectory(prefix="fs_timing_") as work, contextlib.redirect_stdout(io.StringIO()):
        with T.stage("setup"):
            fs = Spectra(SNAP, f"{DATA}/{SIM}", cofm.copy(), axis.copy(), res=1.0, savefile="t.hdf5", savedir=work,
                         reload_file=True, quiet=True)
        with T.patched((fs.snapshot_set, "get_data", "io"), (fs, "_do_interpolation_work", "tau")):
            with T.stage("physics"):
                tau_fs = np.stack([np.asarray(fs.get_tau("H", 1, 1215)), np.asarray(fs.get_tau("Si", 3, 1206))])
    fs_runs.append(dict(T.t))
    print(f"run {run}: " + ", ".join(f"{k} {t:.1f} s" for k, t in T.t.items() if t > 0) + f" (page cache {fs_cache[-1]:.0%}: {cache_label(fs_cache[-1])})")
for stage in ("setup", "io", "selection", "physics", "pieces", "tau"):
    clock.add(FS, stage, [r[stage] for r in fs_runs], warmup=False)
print(f"{os.environ['OMP_NUM_THREADS']} OpenMP threads; kernel {fs.kernel_int} (0 = top hat); {fs.nbins} pixels of {fs.dvbin:.6f} km/s")
del fs

## SkewerJAX: one batch of skewers

The calls of notebook 12, every option written, on one batch: the default on the CPU, the same pieces summed on the GPU,
and the matched mode on the CPU. The I/O and the selection are one call (`select_cells_near_paths`): a thread per block
of every file, read through memory maps, a prefilter of distance bounds on a $256^3$ grid around the skewers, and the
exact rule (radius $2.5\,V^{1/3} + ds/2$ from a path point) on the candidates; its own timings split the two (each stage's
share of the wall time, as the threads overlap them). The selection, the I/O and the setup are shared by the three
entries, and the GPU entry takes the gas physics and the pieces of the default: only its $\tau$ sum runs on the GPU.

The matched mode is fake_spectra's per-cell scheme: the pixel average of narrow cells and its early stop `tautail` (each
cell walks out from its line centre until it adds less than $10^{-7}$), evaluated on a window around every cell
(`core.per_cell`).

In [ ]:
MATCHED_CELLS = dict(mass_unit="proton", T_constants="fake_spectra", x_HI_model="fake_sf", cloudy_density="fake_spectra",
                     cloudy_interpolation="cubic")
MATCHED_TAU = dict(velocity_period=V_PERIOD, voigt="harris", detuning="linear", a_eff=a_snap, hubble_flow="linear", aH_ref=AH,
                   b_constants="fake_spectra", pixel_average=True, tautail=1e-7, resolution_tolerance=None)


def skewerjax_batch(sl, run):
    """The three SkewerJAX entries on the skewers ``sl``, each stage timed into ``clock`` for ``run``: tau of each entry."""
    with clock.time(CPU_DEF, "setup", run):
        bundle = SkewerBundle.axis_aligned(
            np.column_stack([X[sl], np.zeros_like(X[sl]), Z[sl]]),   # a point of each skewer [cMpc] (its y is ignored)
            AXIS,                                                     # every skewer along y, across the box once
            L,
            SPACING_KMS / AH,                                         # the point spacing [cMpc]: 0.25 km/s of Hubble flow
            H.cosmology(),
            a_ref=a_snap,                                             # a(s) = a_snap at the middle of the path
        )
    points = bundle.host["points"]                                    # (n, N, 3) [cMpc], float64
    gas, info = select_cells_near_paths(
        reader, points,
        mode="voronoi",          # the radius of the Voronoi deposition (a superset of the top hat's: one selection for both modes)
        max_rows=4_000_000,      # cells per block, one block per thread
        workers=-1,              # threads: every core
        return_info=True,
    )
    clock.record(CPU_DEF, "io", run, info["timings"]["read"] + info["timings"]["read_selected"])
    clock.record(CPU_DEF, "selection", run, info["timings"]["prefilter"] + info["timings"]["query"])
    # SkewerJAX default
    with clock.time(CPU_DEF, "physics", run):
        cells = gas_to_cells(
            gas, SPECIES,
            mass_unit="amu",
            T_constants="codata",
            x_HI_model="stored_sf",
            cloudy_density="hydrogen",
            cloudy_interpolation="linear",
        )
    with clock.time(CPU_DEF, "pieces", run):
        pieces = cell_pieces(
            cells, lines,
            skewer_bundle=bundle,
            deposition="voronoi",    # straight skewers: the exact lower envelope of the cells along each
            a_data=a_snap,
        )
    default_tau = dict(
        velocity_period=False,
        voigt="harris",              # the default: Harris' expansion (fourth order for Si III), ExoJAX above a = 0.05
        detuning="exact",
        a_eff=None,
        hubble_flow="exact",
        aH_ref=None,
        b_constants="codata",
        pixel_average=False,
        tautail=None,
        resolution_tolerance=0.25,
        piece_chunk=2048,
        dtype=None,                  # the package's precision: float32
    )
    with clock.time(CPU_DEF, "tau", run):
        tau_def = np.asarray(compute_optical_depth(pieces, v, lines, devices=CPU, **default_tau))   # np.asarray waits for the result
    tau_gpu = None
    if GPU:
        with clock.time(GPU_DEF, "tau", run):
            tau_gpu = np.asarray(compute_optical_depth(pieces, v, lines, devices=GPU, **default_tau))
    del pieces, cells
    # SkewerJAX matched
    with clock.time(CPU_MAT, "physics", run):
        cells_m = gas_to_cells(gas, SPECIES, **MATCHED_CELLS)
    frame = SnapshotFrame(a_snap, AH, path_length=np.full(sl.stop - sl.start, L))    # one a and u = aH s + v_pec
    with clock.time(CPU_MAT, "pieces", run):
        pieces_m = cell_pieces(
            cells_m, lines,
            points=points,
            frame=frame,
            deposition="tophat_chords",
            chord_weight=1 / 8,
            a_data=a_snap,
        )
    with clock.time(CPU_MAT, "tau", run):
        tau_mat = np.asarray(compute_optical_depth(pieces_m, v, lines, devices=CPU, **MATCHED_TAU))
    return tau_def, tau_mat, tau_gpu

## The runs

Run 0, the warm-up, is the first batch; then `N_RUNS` timed runs over every batch. The $\tau$ of every skewer is kept from
the last timed run for the checks below.

In [ ]:
TAU = {e: np.empty((2, N_SK, nbins), np.float32) for e in (CPU_DEF, CPU_MAT, GPU_DEF)}
sj_cache = cache_state(SNAPDIR)
skewerjax_batch(BATCHES[0], 0)                                         # run 0: the warm-up (first batch)
first = clock.warmup()
batch0 = []                                                            # the first batch of every timed run
for run in range(1, 1 + N_RUNS):
    for b, sl in enumerate(BATCHES):
        out = skewerjax_batch(sl, run)
        if b == 0:
            batch0.append({e: {s: r[run] for s, r in st.items()} for e, st in clock.t.items() if e != FS})
        for e, t in zip((CPU_DEF, CPU_MAT, GPU_DEF), out):
            if t is not None:
                TAU[e][:, sl] = t
    print(f"run {run}: " + ", ".join(f"{e} {sum(r[run] for r in clock.t[e].values()):.0f} s" for e in (CPU_DEF, CPU_MAT, GPU_DEF) if e in clock.t))
for stage in ("setup", "io", "selection"):                             # shared by the three entries
    clock.share(CPU_DEF, stage, CPU_MAT)
if GPU:
    for stage in ("setup", "io", "selection", "physics", "pieces"):   # the host stages of the GPU entry: the default's
        clock.share(CPU_DEF, stage, GPU_DEF)
JIT = {e: first[e]["tau"] - float(np.median([b[e]["tau"] for b in batch0])) for e in (CPU_DEF, CPU_MAT, GPU_DEF) if e in first}

## Fairness and correctness

The four entries compute the same spectra of the same two lines on the same pixels, in float32 (fake_spectra in double).
The checks: the matched mode against fake_spectra (rms and max $\Delta F$ over every pixel of every skewer), the GPU against
the CPU for the default (to float32 rounding), and the default against the matched mode (the known difference of
notebook 12). **The page cache:** the snapshot (349 GB) is larger than the RAM, so fake_spectra, which reads the fields of
every particle once per line, reads from the disk in every run (cold); SkewerJAX reads the positions, masses and
densities of every cell and then the selected rows only: the fraction of the snapshot in the page cache before each code
reads it is printed below.

In [ ]:
F = lambda t: np.exp(-np.asarray(t, dtype=np.float64))
rms = lambda a, b: float(np.sqrt(np.mean((F(a) - F(b)) ** 2)))
mx = lambda a, b: float(np.max(np.abs(F(a) - F(b))))
for j, line in enumerate(lines.lines):
    print(f"{line.id:11s} matched (CPU) vs fake_spectra: rms dF {rms(TAU[CPU_MAT][j], tau_fs[j]):.1e}, max {mx(TAU[CPU_MAT][j], tau_fs[j]):.1e}"
          f" | default vs matched: rms dF {rms(TAU[CPU_DEF][j], TAU[CPU_MAT][j]):.1e}")
    if GPU:
        print(f"{'':11s} default, GPU vs CPU: rms dF {rms(TAU[GPU_DEF][j], TAU[CPU_DEF][j]):.1e}, max {mx(TAU[GPU_DEF][j], TAU[CPU_DEF][j]):.1e}")
print(f"page cache before the reads: fake_spectra " + ", ".join(f"{c:.0%}" for c in fs_cache) + f" ({cache_label(max(fs_cache))}); "
      f"SkewerJAX {sj_cache:.0%} ({cache_label(sj_cache)})")
print("JIT compilation of the tau sum (warm-up batch minus the same batch timed): " + ", ".join(f"{e} {c:.2f} s" if c > 0 else f"{e} below the run-to-run noise" for e, c in JIT.items()))
print()
print(clock.table())

## Where the time goes

The median wall time of one run: (a) split into its stages, with the total and the JIT compilation of the warm-up (not
in the bars) after each bar; (b) each stage's share; (c) the totals on a log axis with the speed-up over fake_spectra.

In [ ]:
order = [FS, CPU_DEF, CPU_MAT] + ([GPU_DEF] if GPU else [])
med = clock.medians()
fig, axs = plot_stage_grid([{e: med[e] for e in order}], jit=[JIT], figsize=(22, 6.5))
add_panel_letters(axs[0])
fig.suptitle("SkewerJAX's time goes to its τ sum on the CPU, to its host stages on the GPU", fontsize=21)
plt.show()
totals = clock.totals()
for e in order:
    print(f"{e:24s} {totals[e]:8.1f} s" + ("" if e == FS else f": {totals[FS] / totals[e]:5.2f}x fake_spectra's speed"))

## Discussion

**Totals.** fake_spectra computes the 16,384 spectra in **393 s** per run; SkewerJAX takes **2,989 s** on the CPU
($0.13\times$ fake_spectra's speed), **4,365 s** in the matched mode ($0.09\times$) and **1,265 s** with its $\tau$ sum on the GPU ($0.31\times$).
The three timed runs agree to within 3% (the GPU's $\tau$ sum to 0.1%, fake_spectra to 0.4%). The JIT compilation of the warm-up is 2.6 s (CPU) and 2.5 s (GPU), under 0.3% of a run; in the
matched mode it is below the run-to-run noise. Both codes read from the disk: 31% of the snapshot was in the page cache.

**fake_spectra** spends 17% of its run reading (66 s), 35% on the gas physics of every particle (138 s) and 48% on its
search, deposition and $\tau$ sum (189 s, 11.5 ms per skewer on 32 OpenMP threads).

**SkewerJAX on the CPU** spends 68% of its run in the $\tau$ sum (2,026 s, 0.12 s per skewer): the Voigt profile of every
piece over its window of pixels, in XLA, $11\times$ fake_spectra's search, deposition and sum together. The pieces (the exact
lower envelope, 594 s, 36 ms per skewer) take 20%; the cell selection 181 s; the reads 119 s (each batch reads the
positions of every cell again); the gas physics, of the selected cells only, 48 s, a third of fake_spectra's.

**On the GPU** the $\tau$ sum takes 303 s, $6.7\times$ faster than on the CPU; the host stages (962 s) are then 76% of the run,
the pieces alone 47%.

**The matched mode** deposits fake_spectra's top-hat chords: 2,148 s for the pieces ($3.6\times$ the envelope) and 1,841 s for
the $\tau$ sum. It reproduces fake_spectra to rms $\Delta F$ = 1.5e-7 ($\mathrm{Ly}\alpha$) and 9.7e-8 (Si III), max 9.9e-5 and
3.4e-5 over every pixel of every skewer. The GPU agrees with the CPU to float32 rounding (rms $\Delta F \le$ 1.2e-6, max
2.7e-4); the default differs from the matched mode by rms $\Delta F$ = 7.8e-2 ($\mathrm{Ly}\alpha$) and 2.1e-2 (Si III), the
differences of physics of notebook 12.

**Where the gap is.** On the CPU, the $\tau$ sum; on the GPU, the host stages, first the pieces and the cell selection.
How each stage grows with the number of skewers, the snapshot, the pixel width and the sampling is in notebook 14.